In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
import torch.nn.functional as F

from tokenizers.models import BPE
from tokenizers import Tokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

In [ ]:
from pathlib import Path

In [ ]:
path = Path("/kaggle/input/datasets/phnguyencs/chinese-vi/dataset")

In [ ]:
train_path = path / "train"
public_path =path / "public_test"
private_path = path / "private_test"

In [ ]:
zh, vi = [], []
with open(train_path / "train.zh", "r", encoding="utf-8") as f:
    zh = f.read().splitlines()

with open(train_path / "train.vi", "r", encoding="utf-8") as f:
    vi = f.read().splitlines()

print(len(zh), len(vi))
for i in range(5):
    print(zh[i], vi[i])

In [ ]:
import pandas as pd

df= pd.DataFrame({
    "zh": zh,
    "vi": vi
}
)

In [ ]:
df.sample(5)

## Tokenizer

In [ ]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

In [ ]:
trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens = ["[SOS]", "[EOS]", "[PAD]", "[UNK]", "[2ZH]", "[2VI]"],
    min_frequency = 5
)

def _yield(data):
    for sample in data:
        yield(sample)

bilingual_data = zh + vi


In [ ]:
len(bilingual_data)

In [ ]:
tokenizer.train_from_iterator(_yield(bilingual_data), trainer)

In [ ]:
vocab = tokenizer.get_vocab()

print(len(vocab))

In [ ]:
print(list(vocab.keys())[:10])

In [ ]:

PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
SOS_TOKEN_ID = tokenizer.token_to_id("[SOS]")
EOS_TOKEN_ID = tokenizer.token_to_id("[EOS]")

print(
    (PAD_TOKEN_ID, SOS_TOKEN_ID, EOS_TOKEN_ID)
)

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns




sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

zh_count = df['zh'].apply(lambda x: len(tokenizer.encode(x).ids)).tolist()
vi_count = df['vi'].apply(lambda x: len(tokenizer.encode(x).ids)).tolist()
ax[0].hist(zh_count, range=(0,40), bins=8)
ax[0].set_title("zh")
ax[1].hist(vi_count, range=(0,40), bins=8)
ax[1].set_title("vi")

In [ ]:
tokenizer.post_processor = TemplateProcessing(
    single = "[SOS] $A [EOS]",
    special_tokens=[
       ("[SOS]" , SOS_TOKEN_ID),
       ("[EOS]" , EOS_TOKEN_ID),
    ]
)

## Loading Data 

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, zh, vi, tokenizer, isTest=True):
        self.zh = zh
        self.vi = vi
        self.tokenizer = tokenizer
        self.isTest = isTest
        self.samples = []
        if not isTest:
            for i, (zh, vi) in enumerate(zip(self.zh, self.vi)):
                if i % 2 == 0:
                    self.samples.append((self.add_token(zh, "[2VI]"), vi, "zh2vi"))
                else:
                    self.samples.append((self.add_token(vi, "[2ZH]"), zh, "vi2zh"))
        else:
            for i, (zh, vi) in enumerate(zip(self.zh, self.vi)):
                self.samples.append((self.add_token(zh, "[2VI]"), vi, "zh2vi"))
    
        self.zh2vi_indices = [idx for idx, (_, _, d) in enumerate(self.samples) if d == "zh2vi"]
        self.vi2zh_indices = [idx for idx, (_, _, d) in enumerate(self.samples) if d == "vi2zh"]
    def add_token(self, text, token):
        return f"{token} {text}"
            
                    
    def __len__(self) :
       return len(self.samples) 
    def __getitem__(self,idx):
        s1, s2, tag = self.samples[idx]        
        src =  self.tokenizer.encode(s1).ids
        src = torch.tensor(src).squeeze(0)
    
        tgt = self.tokenizer.encode(s2).ids
        tgt = torch.tensor(tgt).squeeze(0)
        tgt_input = tgt[:-1]        
        tgt_output = tgt[1:]        
        return {
            "src": src,
            "tgt_input": tgt_input,
            "tgt_output": tgt_output
        }

In [ ]:
df_train = df.sample(int(0.95 * len(df)), random_state=42)
df_val = df.drop(index=df_train.index)
len(df_train), len(df_val)

In [ ]:
train_data = CustomDataset(df_train['zh'].tolist(), df_train['vi'].tolist(), 
                           tokenizer, False)
val_data = CustomDataset(df_val['zh'].tolist(), df_val['vi'].tolist(), 
                           tokenizer)

In [ ]:
train_data[10]

In [ ]:
MAX_LEN = 25
def collate_fn(batch):
    src = [b["src"][:MAX_LEN ] for b in batch]
    tgt_input = [b["tgt_input"][:MAX_LEN ] for b in batch]
    tgt_output= [b["tgt_output"][:MAX_LEN ] for b in batch]

    src = nn.utils.rnn.pad_sequence(
        sequences=src, batch_first=True, padding_value=PAD_TOKEN_ID
    )
    tgt_input = nn.utils.rnn.pad_sequence(
        sequences=tgt_input, batch_first=True, padding_value=PAD_TOKEN_ID
    )
    tgt_output = nn.utils.rnn.pad_sequence(
        sequences=tgt_output, batch_first=True, padding_value=PAD_TOKEN_ID
    )

    src_key_padding_mask = src == PAD_TOKEN_ID
    tgt_key_padding_mask = tgt_input == PAD_TOKEN_ID
    return {
       "src" : src,
       "tgt_input": tgt_input,
       "tgt_output": tgt_output,
       "src_key_padding_mask": src_key_padding_mask,
       "tgt_key_padding_mask": tgt_key_padding_mask,
    }

In [ ]:
def select_vi2zh_window(indices , epoch, ratio): 
    if not indices or ratio <= 0:
        return []
    total = len(indices)
    window = max(1, int(math.ceil(total * min(ratio, 1.0))))
    start = ((epoch - 1) * window) % total
    end = start + window
    if end <= total:
        return indices[start:end]
    wrap = end - total
    return indices[start:] + indices[:wrap]

In [ ]:
BATCH_SIZE = 128
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=False)

In [ ]:
VI2ZH_EPOCH_RATIO  = 0.7

In [ ]:
def build_train_loader(epoch ): 
    active = list(train_data.zh2vi_indices)
    vi_slice = select_vi2zh_window(train_data.vi2zh_indices, epoch, VI2ZH_EPOCH_RATIO)
    active.extend(vi_slice)
    sampler = SubsetRandomSampler(active)
    loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        collate_fn=collate_fn,
    )
    return loader


## Model

In [ ]:
def create_mask(size, device):
    mask = torch.full((size, size), fill_value=float("-inf"), device=device)
    mask = torch.triu(mask, diagonal=1)
    return mask


In [ ]:
create_mask(4, 'cpu')

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps = 1e-6):
        super().__init__()
        self.dim=  dim
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = x.pow(2).mean(dim = -1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.scale

In [ ]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim =-1)
    return torch.cat((-x2, x1), dim=  -1)

In [ ]:
def rope_cache(dim, seq_len, device):
    half_dim = dim // 2
    angle = 1 / (10000 ** (torch.arange(0, half_dim, device=  device)/half_dim))
    positions = torch.arange(0, seq_len, device=device)
    angle = torch.outer(positions, angle)[None, None, :,  :]
    sin = torch.repeat_interleave(angle.sin(), 2, dim=-1)
    cos = torch.repeat_interleave(angle.cos(), 2, dim=-1)
    return sin, cos

In [ ]:
def apply_rope(x, sin , cos):
    return x * cos + rotate_half(x) * sin

In [ ]:
class Attension(nn.Module):
    def __init__(self, d_model, nhead, dropout ):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = self.d_model // self.nhead
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.o = nn.Linear(d_model, d_model)
    def forward(self, q, k, v, attn_mask, key_padding_mask ):
        batch_size, q_len, _ = q.size()
        k_len = k.size(1)
        q = self.q(q).view(batch_size, q_len, self.nhead, self.head_dim).transpose(1,2)
        k = self.k(k).view(batch_size, k_len, self.nhead, self.head_dim).transpose(1,2)
        v = self.v(v).view(batch_size, k_len, self.nhead, self.head_dim).transpose(1,2)
        
        sin, cos  = rope_cache(self.head_dim, max(q_len, k_len), q.device)
        q = apply_rope(q, sin[:, :, :q_len, :], cos[:, :, :q_len, :])
        k = apply_rope(k, sin[:, :, :k_len, :], cos[:, :, :k_len, :])
        weight = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if attn_mask is not None:
            weight = weight + attn_mask[None, None, :, :]
        if key_padding_mask is not None:
            weight = torch.masked_fill(weight, key_padding_mask[:, None, None, :], float('-inf'))
        weight=  torch.softmax(weight, dim=-1)
        weight = self.dropout(weight)
        out = torch.matmul(weight, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, q_len, self.d_model)
        out = self.o(out)
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, ffn, dropout):
        super().__init__()
        self.w1 = nn.Linear(dim, ffn)
        self.w2 = nn.Linear(dim, ffn)
        self.w3 = nn.Linear(ffn, dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = nn.functional.silu(
            self.w1(x)) * self.w2(x)
        
        return self.w3(self.dropout(x))

In [ ]:
class Encoder(nn.Module):
    def __init__(self, dim, nhead, ffn, dropout ):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.attn = Attension(dim, nhead, dropout)
        self.feed = FeedForward(dim, ffn, dropout)

        
    def forward(self, x, src_key_padding_mask):
        x = x + self.dropout(self.attn(
           self.norm1(x),
           self.norm1(x),
           self.norm1(x),
            None, src_key_padding_mask
        ))
        x = x + self.dropout(self.feed(self.norm2(x)))
        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, dim, nhead, ffn, dropout):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.norm3 = RMSNorm(dim)
        self.attn1 = Attension(dim, nhead, dropout)
        self.attn2 = Attension(dim, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.feed = FeedForward(dim, ffn, dropout)
    def forward(self, memory, memory_key_padding_mask, tgt, tgt_mask, tgt_key_padding_mask):
        tgt = tgt + self.dropout(self.attn1(
           self.norm1(tgt),
           self.norm1(tgt),
           self.norm1(tgt),
            tgt_mask, tgt_key_padding_mask
        ))
        tgt = tgt + self.dropout(self.attn2(
           self.norm2(tgt),
           memory,
           memory,
            None, memory_key_padding_mask
        ))
        tgt = tgt  + self.dropout(self.feed(self.norm3(tgt)))
        return tgt

In [ ]:
import math

In [ ]:
class NMT(nn.Module):
    def __init__(self, dim, nhead, size, dropout, ffn, num_encoders, num_decoders):
        super().__init__()
        self.scale = math.sqrt(dim)
        self.embedd = nn.Embedding(size, dim)
        self.encoder = nn.ModuleList(
            Encoder(dim, nhead, ffn, dropout) for _ in range(num_encoders)
        )
        self.decoder = nn.ModuleList(
            Decoder(dim, nhead, ffn, dropout) for _ in range(num_decoders)
        )
        self.dropout = nn.Dropout(dropout)
        self.norm = RMSNorm(dim)
        self.head=  nn.Linear(dim, size, bias= False)
    def forward(self, src, src_key_padding_mask,
                tgt, tgt_mask, tgt_key_padding_mask):
        src = self.dropout(self.embedd(src) * self.scale)
        memory = src
        for layer in self.encoder:
            memory = layer(memory, src_key_padding_mask)
        
        tgt = self.dropout(self.embedd(tgt) * self.scale)
        for layer in self.decoder:
            tgt = layer(memory, src_key_padding_mask, tgt, tgt_mask, tgt_key_padding_mask)
        out = self.head(self.norm(tgt))
        return out.permute(0, 2, 1)

In [ ]:
DIM = 768
NHEAD = 6
SIZE = 8000
NUM_ENCODERS = 8
NUM_DECODERS = 8
FFN = 3072
DROPOUT = 0.2

In [ ]:
model = NMT(DIM, NHEAD, SIZE, DROPOUT, FFN, NUM_ENCODERS, NUM_DECODERS )

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(DEVICE)

In [ ]:
total_params = 0
for p in model.parameters():
    total_params += p.numel()
print(total_params)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN_ID, label_smoothing=0.05)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    weight_decay=1e-2 ,
)

## Training

In [ ]:
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

In [ ]:

def train(epoch, epochs, train_loader, scheduler):
    model.train()
    correct, count, losses = 0, 0, []

    for inputs in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]"):
        src = inputs["src"].to(DEVICE)
        tgt_input = inputs["tgt_input"].to(DEVICE)
        tgt_output = inputs["tgt_output"].to(DEVICE)
        src_key_padding_mask = inputs["src_key_padding_mask"].to(DEVICE)
        tgt_key_padding_mask = inputs["tgt_key_padding_mask"].to(DEVICE)
        tgt_mask = create_mask(tgt_input.size(1), DEVICE)

        optimizer.zero_grad(set_to_none=True)
        output = model(
            src,
            src_key_padding_mask,
            tgt_input,
            tgt_mask,
            tgt_key_padding_mask
        )
        loss = criterion(output, tgt_output)
        losses.append(loss.item())
        pred = output.argmax(1)
        non_pad_mask = tgt_output != PAD_TOKEN_ID
        correct += (pred[non_pad_mask] == tgt_output[non_pad_mask]).sum().item()
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        count += non_pad_mask.sum().item()

    epoch_loss = sum(losses) / len(losses)
    epoch_acc = correct / count
    return epoch_acc, epoch_loss


In [ ]:
def evaluate(epoch, epochs, val_loader):
    model.eval()
    correct, count, losses = 0, 0, []
    with torch.no_grad():
        for inputs in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
            src = inputs["src"].to(DEVICE)
            tgt_input = inputs["tgt_input"].to(DEVICE)
            tgt_output = inputs["tgt_output"].to(DEVICE)
            src_key_padding_mask = inputs["src_key_padding_mask"].to(DEVICE)
            tgt_key_padding_mask = inputs["tgt_key_padding_mask"].to(DEVICE)
            tgt_mask = create_mask(tgt_input.size(1), DEVICE)

            output = model(
                src,
                src_key_padding_mask,
                tgt_input,
                tgt_mask,
                tgt_key_padding_mask
            )
            loss = criterion(output, tgt_output)
            losses.append(loss.item())
            pred = output.argmax(1)
            non_pad_mask = tgt_output != PAD_TOKEN_ID
            correct += (pred[non_pad_mask] == tgt_output[non_pad_mask]).sum().item()
            count += non_pad_mask.sum().item()

    epoch_loss = sum(losses) / len(losses)
    epoch_acc = correct / count
    return epoch_acc, epoch_loss


In [ ]:
!pip install sacrebleu

In [ ]:
import sacrebleu
@torch.no_grad()
def compute_bleu(zhs, vis):
    preds = []
    for zh in zhs:
        zh = f"[2VI] {zh}"
        src = tokenizer.encode(zh).ids
        src = torch.tensor(src, device=DEVICE).unsqueeze(0)
        src = src[:, :MAX_LEN]
        src_key_padding_mask = src == PAD_TOKEN_ID
        
        ys = torch.tensor([[SOS_TOKEN_ID]], dtype=torch.long, device=DEVICE)
        max_len = src.size(1) + 5

        for i in range(max_len - 1):
            tgt_mask = create_mask(ys.size(1), DEVICE)
            tgt_key_padding_mask = ys == PAD_TOKEN_ID
            
            output = model(
                src,
                src_key_padding_mask,
                ys,
                tgt_mask,
                tgt_key_padding_mask
            )
            pred = output[:, :, -1].argmax(1).item()
            ys = torch.cat(
                (ys, torch.tensor([[pred]], dtype = torch.long, device=DEVICE,)),  dim =-1
            )
            
            if pred == EOS_TOKEN_ID:
                break 
            
        pred = tokenizer.decode(ys.squeeze(0).tolist(), skip_special_tokens=True)
        preds.append(pred)
    bleu = sacrebleu.corpus_bleu(preds, [vis], force=True)
    return bleu.score

In [ ]:

train_losses, val_losses, train_accs, val_accs, bleu_scores = [], [], [], [], []

In [ ]:
min_loss = 10

In [ ]:
def training(epochs, test_loader, bleu_every=2):
    train_loader  = build_train_loader(1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-4,
        total_steps=epochs * len(train_loader),
        pct_start=0.1,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0,
    )
    model_path = "./model.pth"
    tokenizer_path = "./tokenizer.json"
    tokenizer.save(tokenizer_path)

    for epoch in range(1, epochs+1):
        train_loader  = build_train_loader(epoch)
        train_acc, train_loss = train(epoch, epochs, train_loader, scheduler)
        val_acc, val_loss = evaluate(epoch, epochs, test_loader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        global min_loss
        if val_loss < min_loss:
            min_loss = val_loss
            torch.save(model.state_dict(), model_path)
            
        bleu_score = None
        if epoch % bleu_every == 0 or epoch == epochs - 1:
            bleu_score = compute_bleu(
                df_val["zh"].iloc[:100].tolist(),
                df_val["vi"].iloc[:100].tolist(),
            )
            bleu_scores.append(bleu_score)

        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
        if bleu_score is None:
            print("  Val BLEU: skipped\n")
        else:
            print(f"  Val BLEU: {bleu_score:.2f}\n")


In [ ]:

training(
    epochs=40, test_loader=val_loader
)

In [ ]:

fig, axs = plt.subplots(nrows = 1, ncols =2 , figsize = (12,6))
epochs = list(range(len(train_accs)))
axs[0].plot(epochs,train_accs, label = "Training")
axs[0].plot(epochs, val_accs, label = "Evaluation")
axs[1].plot(epochs, train_losses, label = "Training")
axs[1].plot(epochs, val_losses, label = "Evaluation")
axs[0].set_xlabel("Epochs")
axs[1].set_xlabel("Epochs")
axs[0].set_ylabel("Accuracy")
axs[1].set_ylabel("Loss")
plt.legend()
plt.savefig('./training_curves.png')
plt.show()

In [ ]:
epochs = list(range(len(bleu_scores)))
plt.plot(epochs, bleu_scores)
plt.xlabel("Epochs x2")
plt.ylabel("BLEU Score")
plt.savefig('./bleu_scores.png')
plt.show()

In [ ]:
best_model = NMT(DIM, NHEAD, SIZE, DROPOUT, FFN, NUM_ENCODERS, NUM_DECODERS )
best_model.load_state_dict(torch.load("/kaggle/working/model.pth", weights_only=True))
best_model = best_model.to(DEVICE)
best_model

In [ ]:
@torch.no_grad()
def inference(zhs):
    preds = []
    model.eval()
    for zh in zhs:
        zh = f"[2VI] {zh}"
        src = tokenizer.encode(zh).ids
        src = torch.tensor(src, device=DEVICE).unsqueeze(0)
        src = src[:, :MAX_LEN]
        src_key_padding_mask = src == PAD_TOKEN_ID
        
        ys = torch.tensor([[SOS_TOKEN_ID]], dtype=torch.long, device=DEVICE)
        max_len = src.size(1) + 5

        for i in range(max_len - 1):
            tgt_mask = create_mask(ys.size(1), DEVICE)
            tgt_key_padding_mask = ys == PAD_TOKEN_ID
            
            output = model(
                src,
                src_key_padding_mask,
                ys,
                tgt_mask,
                tgt_key_padding_mask
            )
            pred = output[:, :, -1].argmax(1).item()
            ys = torch.cat(
                (ys, torch.tensor([[pred]], dtype = torch.long, device=DEVICE,)),  dim =-1
            )
            
            if pred == EOS_TOKEN_ID:
                break 
            
        pred = tokenizer.decode(ys.squeeze(0).tolist(), skip_special_tokens=True)
        preds.append(pred)
    return preds

In [ ]:
zh_private = []
with open(private_path / "private_test.zh", "r", encoding="utf-8") as f:
     zh_private = f.read().splitlines()
vi_private = inference(zh_private)

In [ ]:
len(zh_private), len(vi_private)

In [ ]:
zh_private[1], vi_private[1]

In [ ]:
public_submit = pd.DataFrame({
    "tieng_trung": zh_private,
    "tieng_viet": vi_private
})

In [ ]:
public_submit.sample(5)

In [ ]:
public_submit.to_csv("nlp_submission.csv", index=False)

In [ ]:
!zip -r "nlp.zip" nlp_submission.csv